# 02 - Feature extraction

Questo notebook trasforma i segnali grezzi WESAD in un dataset tabellare di feature.

La versione usa una pipeline semplice ma completa:

- usa i segnali `chest`;
- tiene solo le label `1`, `2`, `3`;
- applica il filtraggio esplicitamente citato nel paper WESAD per `EDA` e `Resp`;
- lascia `ECG`, `EMG` e `Temp` senza filtro esplicito;
- usa finestre da 60 secondi senza overlap;
- calcola feature statistiche base;
- salva un file CSV da usare nei notebook di classificazione.


## 1. Import e configurazione


In [1]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from scipy.signal import butter, sosfiltfilt
except ImportError as exc:
    raise ImportError(
        "Questo notebook richiede scipy per applicare i filtri Butterworth. "
        "Su Kaggle scipy e' normalmente gia' disponibile."
    ) from exc

try:
    import kagglehub
except ImportError:
    kagglehub = None

LABEL_NAMES = {
    1: "baseline",
    2: "stress",
    3: "amusement",
}

LABELS_TO_KEEP = set(LABEL_NAMES.keys())
CHEST_SIGNALS = ["ECG", "EDA", "EMG", "Resp", "Temp"]
CHEST_SAMPLING_RATE = 700
WINDOW_SECONDS = 60
STEP_SECONDS = 60
MIN_LABEL_FRACTION = 0.8
MAX_SUBJECTS = None

FILTER_CONFIG = {
    "EDA": {"type": "lowpass", "cutoff": 5, "order": 4},
    "Resp": {"type": "bandpass", "cutoff": [0.1, 0.35], "order": 2},
}

print("Segnali usati:", CHEST_SIGNALS)
print("Finestra:", WINDOW_SECONDS, "secondi")
print("Step:", STEP_SECONDS, "secondi")
print("Filtri applicati, come indicato nel paper WESAD:")
for signal_name, config in FILTER_CONFIG.items():
    print(" ", signal_name, config)

print("Segnali lasciati senza filtro esplicito:", [s for s in CHEST_SIGNALS if s not in FILTER_CONFIG])


Segnali usati: ['ECG', 'EDA', 'EMG', 'Resp', 'Temp']
Finestra: 60 secondi
Step: 60 secondi
Filtri applicati, come indicato nel paper WESAD:
  EDA {'type': 'lowpass', 'cutoff': 5, 'order': 4}
  Resp {'type': 'bandpass', 'cutoff': [0.1, 0.35], 'order': 2}
Segnali lasciati senza filtro esplicito: ['ECG', 'EMG', 'Temp']


## 2. Trovare il dataset WESAD


In [2]:
KAGGLE_INPUT_PATH = Path("/kaggle/input")
LOCAL_DATASET_PATH = Path("data/raw/WESAD")
KAGGLE_DATASET_SLUG = "mohamedasem318/wesad-full-dataset"


def find_wesad_path():
    expected_kaggle_path = Path("/kaggle/input/datasets/mohamedasem318/wesad-full-dataset/WESAD")
    if expected_kaggle_path.exists():
        return expected_kaggle_path

    if KAGGLE_INPUT_PATH.exists():
        matches = list(KAGGLE_INPUT_PATH.rglob("WESAD"))
        if matches:
            return matches[0]

    if LOCAL_DATASET_PATH.exists():
        return LOCAL_DATASET_PATH

    if kagglehub is not None:
        downloaded_path = Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG))
        wesad_path = downloaded_path / "WESAD"
        if wesad_path.exists():
            return wesad_path

    raise FileNotFoundError("Dataset WESAD non trovato. Collegalo su Kaggle oppure imposta LOCAL_DATASET_PATH.")


def subject_sort_key(path):
    return int(path.name[1:]) if path.name.startswith("S") and path.name[1:].isdigit() else 999


def list_subject_paths(dataset_path):
    subject_paths = []
    for path in dataset_path.iterdir():
        if path.is_dir() and path.name.startswith("S") and (path / f"{path.name}.pkl").exists():
            subject_paths.append(path)
    return sorted(subject_paths, key=subject_sort_key)


def load_subject(subject_path):
    pkl_path = subject_path / f"{subject_path.name}.pkl"
    with open(pkl_path, "rb") as f:
        return pickle.load(f, encoding="latin1")


DATASET_PATH = find_wesad_path()
SUBJECT_PATHS = list_subject_paths(DATASET_PATH)

print("DATASET_PATH:", DATASET_PATH)
print("Soggetti trovati:", [p.name for p in SUBJECT_PATHS])

if MAX_SUBJECTS is not None:
    SUBJECT_PATHS = SUBJECT_PATHS[:MAX_SUBJECTS]

print("Numero soggetti usati:", len(SUBJECT_PATHS))


DATASET_PATH: /kaggle/input/datasets/mohamedasem318/wesad-full-dataset/WESAD
Soggetti trovati: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
Numero soggetti usati: 15


## 3. Controllo veloce delle label per soggetto

Prima di estrarre le feature controlliamo che i soggetti abbiano le label principali del progetto.


In [3]:
summary_rows = []

for subject_path in SUBJECT_PATHS:
    data = load_subject(subject_path)
    labels = data["label"]
    counts = pd.Series(labels).value_counts().to_dict()
    row = {"subject": subject_path.name, "n_samples": len(labels)}
    for label_id, label_name in LABEL_NAMES.items():
        row[label_name] = counts.get(label_id, 0)
    summary_rows.append(row)

label_summary = pd.DataFrame(summary_rows)
label_summary


,subject,n_samples,baseline,stress,amusement
0,S2,4255300,800800,430500,253400
1,S3,4545100,798000,448000,262500
2,S4,4496100,810601,444500,260400
3,S5,4380600,838600,451500,261800
4,S6,4949700,826000,455000,260400
5,S7,3666600,830200,448000,260401
6,S8,3826200,818300,469000,258999
7,S9,3656100,826000,451500,260400
8,S10,3847200,826000,507500,260400
9,S11,3663100,826000,476000,257600


## 4. Funzioni di filtraggio e feature extraction

Il filtraggio viene applicato prima della divisione in finestre.

Per restare aderenti al paper WESAD, in questa versione applichiamo solo i filtri indicati esplicitamente dagli autori:

- `EDA`: low-pass 5 Hz;
- `Resp`: band-pass 0.1-0.35 Hz.

Gli altri segnali usati nella pipeline (`ECG`, `EMG`, `Temp`) vengono lasciati senza filtro esplicito. Questa scelta evita di introdurre parametri non direttamente motivati dal paper.


In [4]:
def get_output_dir():
    if Path("/kaggle/working").exists():
        output_dir = Path("/kaggle/working/data/processed")
    elif Path.cwd().name == "notebooks":
        output_dir = Path("../data/processed")
    else:
        output_dir = Path("data/processed")

    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def butterworth_filter(values, filter_type, cutoff, fs, order):
    values = np.asarray(values, dtype=float).reshape(-1)
    nyquist = fs / 2
    normalized_cutoff = np.asarray(cutoff, dtype=float) / nyquist

    sos = butter(
        N=order,
        Wn=normalized_cutoff,
        btype=filter_type,
        output="sos",
    )

    return sosfiltfilt(sos, values)


def filter_chest_signals(chest):
    processed = {}

    for signal_name in CHEST_SIGNALS:
        raw_values = np.asarray(chest[signal_name], dtype=float).reshape(-1)

        if signal_name not in FILTER_CONFIG:
            processed[signal_name] = raw_values
            continue

        config = FILTER_CONFIG[signal_name]
        processed[signal_name] = butterworth_filter(
            values=raw_values,
            filter_type=config["type"],
            cutoff=config["cutoff"],
            fs=CHEST_SAMPLING_RATE,
            order=config["order"],
        )

    return processed


def dominant_label(window_labels):
    values, counts = np.unique(window_labels, return_counts=True)
    best_index = np.argmax(counts)
    label = int(values[best_index])
    fraction = counts[best_index] / len(window_labels)
    return label, fraction


def extract_signal_features(values, prefix):
    values = np.asarray(values).reshape(-1)
    return {
        f"{prefix}_mean": np.mean(values),
        f"{prefix}_std": np.std(values),
        f"{prefix}_min": np.min(values),
        f"{prefix}_max": np.max(values),
        f"{prefix}_range": np.max(values) - np.min(values),
        f"{prefix}_median": np.median(values),
    }


def extract_subject_features(data, subject_id):
    chest = data["signal"]["chest"]
    labels = data["label"]
    filtered_chest = filter_chest_signals(chest)

    window_size = WINDOW_SECONDS * CHEST_SAMPLING_RATE
    step_size = STEP_SECONDS * CHEST_SAMPLING_RATE

    n_samples = min(len(labels), *[len(filtered_chest[signal]) for signal in CHEST_SIGNALS])
    rows = []

    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window_labels = labels[start:end]
        label, label_fraction = dominant_label(window_labels)

        if label not in LABELS_TO_KEEP:
            continue

        if label_fraction < MIN_LABEL_FRACTION:
            continue

        row = {
            "subject": subject_id,
            "start_idx": start,
            "end_idx": end,
            "start_seconds": start / CHEST_SAMPLING_RATE,
            "end_seconds": end / CHEST_SAMPLING_RATE,
            "label": label,
            "label_name": LABEL_NAMES[label],
            "binary_label": "stress" if label == 2 else "non-stress",
            "label_fraction": label_fraction,
        }

        for signal_name in CHEST_SIGNALS:
            signal_values = filtered_chest[signal_name][start:end]
            row.update(extract_signal_features(signal_values, signal_name))

        rows.append(row)

    return rows


## 5. Estrazione delle feature da tutti i soggetti


In [5]:
all_rows = []

for subject_path in SUBJECT_PATHS:
    print("Elaboro", subject_path.name)
    data = load_subject(subject_path)
    subject_rows = extract_subject_features(data, subject_path.name)
    print("  finestre utili:", len(subject_rows))
    all_rows.extend(subject_rows)

features = pd.DataFrame(all_rows)
print("Shape finale:", features.shape)
features.head()


Elaboro S2
  finestre utili: 34
Elaboro S3
  finestre utili: 34
Elaboro S4
  finestre utili: 34
Elaboro S5
  finestre utili: 35
Elaboro S6
  finestre utili: 34
Elaboro S7
  finestre utili: 35
Elaboro S8
  finestre utili: 36
Elaboro S9
  finestre utili: 35
Elaboro S10
  finestre utili: 35
Elaboro S11
  finestre utili: 35
Elaboro S13
  finestre utili: 35
Elaboro S14
  finestre utili: 36
Elaboro S15
  finestre utili: 36
Elaboro S16
  finestre utili: 36
Elaboro S17
  finestre utili: 36
Shape finale: (526, 39)


,subject,start_idx,end_idx,start_seconds,end_seconds,label,label_name,binary_label,label_fraction,ECG_mean,...,Resp_min,Resp_max,Resp_range,Resp_median,Temp_mean,Temp_std,Temp_min,Temp_max,Temp_range,Temp_median
0,S2,210000,252000,300.0,360.0,1,baseline,non-stress,0.890881,0.001466,...,-5.550925,4.511106,10.062031,-0.086697,29.167694,0.060271,29.011780,29.426208,0.414429,29.158386
1,S2,252000,294000,360.0,420.0,1,baseline,non-stress,1.000000,0.000986,...,-1.391944,1.702889,3.094832,0.006293,28.885604,0.081293,28.727814,29.207275,0.479462,28.876862
2,S2,294000,336000,420.0,480.0,1,baseline,non-stress,1.000000,0.001455,...,-4.359904,2.069612,6.429516,0.048712,28.798114,0.037777,28.679108,28.988800,0.309692,28.793732
3,S2,336000,378000,480.0,540.0,1,baseline,non-stress,1.000000,0.001022,...,-1.527553,1.812610,3.340163,-0.008351,28.753808,0.063520,28.584656,29.023285,0.438629,28.736389
4,S2,378000,420000,540.0,600.0,1,baseline,non-stress,1.000000,0.001122,...,-1.513284,1.430502,2.943786,-0.005847,28.573873,0.041704,28.447449,28.820953,0.373505,28.561768


## 6. Controllo del dataset di feature


In [6]:
print("Righe per classe binaria:")
print(features["binary_label"].value_counts())

print("\nRighe per label originale:")
print(features["label_name"].value_counts())

print("\nRighe per soggetto:")
print(features["subject"].value_counts().sort_index())


Righe per classe binaria:
binary_label
non-stress    369
stress        157
Name: count, dtype: int64

Righe per label originale:
label_name
baseline     285
stress       157
amusement     84
Name: count, dtype: int64

Righe per soggetto:
subject
S10    35
S11    35
S13    35
S14    36
S15    36
S16    36
S17    36
S2     34
S3     34
S4     34
S5     35
S6     34
S7     35
S8     36
S9     35
Name: count, dtype: int64


## 7. Salvataggio del CSV


In [7]:
OUTPUT_DIR = get_output_dir()
OUTPUT_PATH = OUTPUT_DIR / "features_chest.csv"

features.to_csv(OUTPUT_PATH, index=False)

print("Feature salvate in:", OUTPUT_PATH)
print("Numero colonne:", len(features.columns))


Feature salvate in: /kaggle/working/data/processed/features_chest.csv
Numero colonne: 39
